In [19]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/credit_default_clean.csv")
df.head()
print(df.shape)

(30000, 25)


In [20]:
## AVERAGE REPAYMENT DELAYS
## helps filter out Chronic payment delays

pay_cols = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]
df["avg_repayment_delay"] = df[pay_cols].mean(axis=1)


In [21]:
## Worst Delinquency
## Credit level 9 - Final stage of credit risk (Above 180 days of no payment)
##

df["max_repayment_delay"] = df[pay_cols].max(axis=1)

#### PAYMENT CONSISTENCY (Discipline Score)

Considering that:
##### avg_repayment_delay -> Used to find the average delays
##### max_repayment_delay -> Used to find the worst delay


We still do not have answer to the following questions:
Does the person miss payments frequently? or just once?

This question helps us by finding a pattern of payment delays and further help us judge the customer.

##### THE IDEA OF THIS SCORE IS:
If repayment status > 0 → that month was late.


In [22]:
# PAYMENT CONSISTENCY ( Discipline Score)

df["late_payment_count"] = (df[pay_cols] > 0).sum(axis=1)

df[["late_payment_count"]].head()

df["late_payment_count"].describe()  # In-depth details on the new column of Discipline in payments

count    30000.000000
mean         0.834200
std          1.554303
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          6.000000
Name: late_payment_count, dtype: float64

In [23]:
# TOTAL BILL AMOUNT

bill_cols = ["bill_amt1", "bill_amt2", "bill_amt3",
             "bill_amt4", "bill_amt5", "bill_amt6"]

df["total_bill_amount"] = df[bill_cols].sum(axis=1)

df["total_bill_amount"].describe()

count    3.000000e+04
mean     2.698617e+05
std      3.795643e+05
min     -3.362590e+05
25%      2.868800e+04
50%      1.263110e+05
75%      3.426265e+05
max      5.263883e+06
Name: total_bill_amount, dtype: float64

In [24]:
# CREDIT UTILIZATION RATIO

df["credit_utilization"] = df["total_bill_amount"] / df["limit_bal"]

df["credit_utilization"].describe()

count    30000.000000
mean         2.238288
std          2.111340
min         -1.395540
25%          0.179982
50%          1.709004
75%          4.127575
max         32.185850
Name: credit_utilization, dtype: float64

In [25]:
# TOTAL PAYMENT AMOUNT

pay_amt_cols = ["pay_amt1","pay_amt2","pay_amt3",
                "pay_amt4","pay_amt5","pay_amt6"]

df["total_payment_amount"] = df[pay_amt_cols].sum(axis=1)

df["total_payment_amount"].describe()

count    3.000000e+04
mean     3.165139e+04
std      6.082768e+04
min      0.000000e+00
25%      6.679750e+03
50%      1.438300e+04
75%      3.350350e+04
max      3.764066e+06
Name: total_payment_amount, dtype: float64

BANK TRUSTS PEOPLE WHO PAY BACK

In [26]:

# PAYMENT TO BILL RATIO

df["payment_to_bill_ratio"] = df["total_payment_amount"] / df["total_bill_amount"]

df["payment_to_bill_ratio"] = df["payment_to_bill_ratio"].replace([np.inf, -np.inf], np.nan)  # Replacing infinity with NaN to fix errors
df["payment_to_bill_ratio"].describe()



count    29130.000000
mean         0.392318
std          7.784430
min       -546.928571
25%          0.042188
50%          0.092133
75%          0.615363
max        797.000000
Name: payment_to_bill_ratio, dtype: float64

### RISK BEHAVIOR INDICATORS


In [28]:
"PAYMENT DEFICIT (DEBT GROWTH)"
' Bills > Payments '

df["payment_deficit"] = df["total_bill_amount"] - df["total_payment_amount"]

df["payment_deficit"].describe()  # Provides the statistical information


count    3.000000e+04
mean     2.382103e+05
std      3.631651e+05
min     -2.671514e+06
25%      4.520750e+03
50%      1.019230e+05
75%      3.057178e+05
max      4.116080e+06
Name: payment_deficit, dtype: float64

In [29]:
"PAYMENT TO CREDIT LIMIT RATIO"
'Measures how aggressively someone repays relative to their credit capacity'
'Measures how much of the credit limit the customer pays back'

df["payment_to_limit_ratio"] = df["total_payment_amount"] / df["limit_bal"]

df["payment_to_limit_ratio"].describe()

count    30000.000000
mean         0.233451
std          0.315753
min          0.000000
25%          0.067714
50%          0.156667
75%          0.263153
max         14.566337
Name: payment_to_limit_ratio, dtype: float64

In [30]:
"BILL TO LIMIT RATIO"
'Measures how much of the credit limit is used'

' <0.3        -> Risk level is Safe '
' 0.3 - 0.7   -> Risk level is Medium '
' >0.7        -> Risk level is High '

df["bill_to_limit_ratio"] = df["total_bill_amount"] / df["limit_bal"]

df["bill_to_limit_ratio"].describe()

count    30000.000000
mean         2.238288
std          2.111340
min         -1.395540
25%          0.179982
50%          1.709004
75%          4.127575
max         32.185850
Name: bill_to_limit_ratio, dtype: float64

In [32]:
"HIGH UTILIZATION FLAG"
' Sometimes ML models respond better with binary features hence using a flag '
' 1 = Risky credit usage'

df["high_utilization_flag"] = (df["bill_to_limit_ratio"] > 0.7).astype(int)
df["high_utilization_flag"].value_counts()

high_utilization_flag
1    18485
0    11515
Name: count, dtype: int64

In [33]:
"CHRONIC LATE PAYER FLAG"
' 1 = Frequently late payments '

df["chronic_late_flag"] = (df["late_payment_count"] >= 3).astype(int)
df["chronic_late_flag"].value_counts()

chronic_late_flag
0    26256
1     3744
Name: count, dtype: int64